## 准备数据

In [17]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [18]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [19]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([784, 128], stddev=0.1), dtype=tf.float32)
        self.b1 = tf.Variable(tf.zeros([128]), dtype=tf.float32)
        self.W2 = tf.Variable(tf.random.normal([128, 10], stddev=0.1), dtype=tf.float32)
        self.b2 = tf.Variable(tf.zeros([10]), dtype=tf.float32)

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 784])
        hidden = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(hidden, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [20]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [21]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.6333802 ; accuracy 0.069866665
epoch 1 : loss 2.6060312 ; accuracy 0.07105
epoch 2 : loss 2.5805373 ; accuracy 0.0727
epoch 3 : loss 2.5566337 ; accuracy 0.07475
epoch 4 : loss 2.5341046 ; accuracy 0.07728333
epoch 5 : loss 2.5127792 ; accuracy 0.07981667
epoch 6 : loss 2.4925172 ; accuracy 0.08313333
epoch 7 : loss 2.4732046 ; accuracy 0.0864
epoch 8 : loss 2.454744 ; accuracy 0.09013333
epoch 9 : loss 2.4370513 ; accuracy 0.09363333
epoch 10 : loss 2.420055 ; accuracy 0.097616665
epoch 11 : loss 2.4036925 ; accuracy 0.10216667
epoch 12 : loss 2.3879116 ; accuracy 0.10665
epoch 13 : loss 2.3726666 ; accuracy 0.11075
epoch 14 : loss 2.3579166 ; accuracy 0.11586667
epoch 15 : loss 2.343626 ; accuracy 0.12126666
epoch 16 : loss 2.3297594 ; accuracy 0.12685
epoch 17 : loss 2.3162844 ; accuracy 0.13263333
epoch 18 : loss 2.303174 ; accuracy 0.1386
epoch 19 : loss 2.2904048 ; accuracy 0.14376667
epoch 20 : loss 2.2779517 ; accuracy 0.15023333
epoch 21 : loss 2.2657995 ; acc